# 04 — Training Demo

This notebook is a thin, interactive wrapper. The real training logic — patch extraction, the train/val loop, checkpointing, early stopping, metrics — lives in `src/train.py`, `src/dataset.py`, and `src/experiment.py`, so it can be run without a notebook at all:

```bash
python -m src.train --epochs 30 --exp-name baseline
```

This notebook just calls the same functions and plots the results inline.

In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.train import run_training, build_dataset
from src.dataset import BioHubDataset
from src.detector import CellDetector


c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Look at one sample before training (sanity check, not required for training)

In [2]:
dataset = BioHubDataset(config.DATASET_PATH)
sample = dataset.train_samples[0]
volume = np.asarray(dataset.load_volume(sample))
projection = volume[0].max(axis=0)

detector = CellDetector(
    sigma=config.GAUSSIAN_SIGMA,
    threshold_abs=config.DETECTION_THRESHOLD,
    min_distance=config.CELL_RADIUS,
)
centers = detector.detect(projection)

plt.figure(figsize=(6, 6))
plt.imshow(projection, cmap="gray")
plt.scatter(centers[:, 1], centers[:, 0], c="red", s=10)
plt.title(f"{sample}: {len(centers)} candidate centers")
plt.show()


FileNotFoundError: Train folder not found: E:\Datasets\BioHub\biohub-cell-tracking-during-development\train

## 2. Run training

This calls `run_training()` in `src/train.py` — the exact same function `python -m src.train` runs from the command line. Every run creates a new `experiments/expNNN/` folder with its config, per-epoch metrics, loss/accuracy plots, confusion matrix, and best checkpoint (see `src/experiment.py`).

In [ ]:
exp = run_training([
    "--epochs", str(config.NUM_EPOCHS),
    "--exp-name", "demo_run",
])

print("Experiment folder:", exp.dir)


## 3. Inspect the results this run saved

In [ ]:
import pandas as pd

metrics_df = pd.read_csv(exp.dir / "metrics.csv")
metrics_df


In [ ]:
from PIL import Image

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ["loss.png", "accuracy.png", "confusion_matrix.png"]):
    ax.imshow(Image.open(exp.dir / name))
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
import json

report = json.load(open(exp.dir / "report.json"))
report
